In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit
import warnings
warnings.filterwarnings('ignore')

df = pd.read_parquet("../../data/processed/pickup_clean.parquet")
print(f"Loaded: {df.shape}")

def ds_to_date(ds):
    return pd.Timestamp(year=2022, month=ds // 100, day=ds % 100)

ds_map = {d: ds_to_date(d) for d in df['ds'].unique()}
df['real_date'] = df['ds'].map(ds_map)

df = df.sort_values(['real_date', 'accept_time']).reset_index(drop=True)

print("Sorted by real_date, accept_time")
print(f"Date range: {df['real_date'].min()} → {df['real_date'].max()}")
print(f"ds range: {df['ds'].min()} → {df['ds'].max()}")

print(f"\nTrain (ds<=950): {(df['ds']<=950).sum():,}")
print(f"Test  (ds>=951): {(df['ds']>=951).sum():,}")

Loaded: (6064908, 18)
Sorted by real_date, accept_time
Date range: 2022-05-01 00:00:00 → 2022-10-31 00:00:00
ds range: 501 → 1031

Train (ds<=950): 4,946,313
Test  (ds>=951): 1,118,595


In [ ]:
is_train = df['ds'] <= 950

df['hour_of_day'] = df['accept_time'].dt.hour

df['expected_duration'] = (df['time_window_end'] - df['time_window_start']).dt.total_seconds() / 60

df['courier_orders_so_far'] = df.groupby(['courier_id', 'ds']).cumcount()

df['day_of_week'] = df['real_date'].dt.dayofweek

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

df['accept_distance_km'] = haversine(
    df['accept_gps_lat'], df['accept_gps_lng'],
    df['lat'], df['lng']
)

print("Simple features built:")
for c in ['hour_of_day','expected_duration','courier_orders_so_far','day_of_week','accept_distance_km']:
    print(f"  {c:25s} nulls={df[c].isnull().sum():>9,}  min={df[c].min():.2f}  max={df[c].max():.2f}")

Simple features built:
  hour_of_day               nulls=        0  min=0.00  max=23.00
  expected_duration         nulls=        0  min=4.00  max=1859.00
  courier_orders_so_far     nulls=        0  min=0.00  max=89.00
  day_of_week               nulls=        0  min=0.00  max=6.00
  accept_distance_km        nulls=2,746,705  min=0.00  max=1833.68


In [ ]:
valid = df['accept_distance_km'].notna()
print("Distance percentiles (km):")
print(df['accept_distance_km'].describe(percentiles=[.5,.9,.99,.999]).to_string())

for thresh in [50, 100, 500]:
    n = (df['accept_distance_km'] > thresh).sum()
    print(f"  Distance > {thresh}km: {n:,} ({n/valid.sum()*100:.3f}% of valid)")

Distance percentiles (km):
count    3.318203e+06
mean     1.334159e+00
std      7.526025e+00
min      0.000000e+00
50%      6.123402e-01
90%      2.978685e+00
99%      8.314211e+00
99.9%    2.304081e+01
max      1.833677e+03
  Distance > 50km: 1,795 (0.054% of valid)
  Distance > 100km: 1,325 (0.040% of valid)
  Distance > 500km: 96 (0.003% of valid)


In [ ]:
before_max = df['accept_distance_km'].max()
df['accept_distance_km'] = df['accept_distance_km'].clip(upper=50)
print(f"Capped distance: was max {before_max:.0f}km, now max {df['accept_distance_km'].max():.0f}km")
print(f"Rows affected: {(df['accept_distance_km'] == 50).sum():,}")

print("\nAfter cap:")
print(df['accept_distance_km'].describe(percentiles=[.5,.9,.99]).to_string())

Capped distance: was max 1834km, now max 50km
Rows affected: 1,795

After cap:
count    3.318203e+06
mean     1.239529e+00
std      2.109689e+00
min      0.000000e+00
50%      6.123402e-01
90%      2.978685e+00
99%      8.314211e+00
max      5.000000e+01


In [ ]:
GLOBAL_RATE = df[is_train]['is_disrupted'].mean()
print(f"Global disruption rate (train only): {GLOBAL_RATE:.5f}")
ALPHA = 20 

def smoothed_rate_map(train_df, group_col, target='is_disrupted', alpha=ALPHA, global_rate=GLOBAL_RATE):
    """Compute Bayesian-smoothed disruption rate per category, from TRAIN data only.
    Returns a Series mapping category -> smoothed rate."""
    agg = train_df.groupby(group_col)[target].agg(['sum', 'count'])
    smoothed = (agg['sum'] + alpha * global_rate) / (agg['count'] + alpha)
    return smoothed

test_map = smoothed_rate_map(df[is_train], 'courier_id')
print(f"\nCourier rate map: {len(test_map):,} couriers")
print(f"Sample smoothed rates:\n{test_map.head()}")
print(f"\nRange: {test_map.min():.5f} → {test_map.max():.5f}")

Global disruption rate (train only): 0.01825

Courier rate map: 14,154 couriers
Sample smoothed rates:
courier_id
1    0.017379
2    0.008902
3    0.064191
4    0.000482
5    0.016589
dtype: float64

Range: 0.00005 → 0.77911


In [ ]:
TARGET = 'is_disrupted'
ALPHA = 20

TE_FEATURES = {
    'courier_late_rate':   ['courier_id'],
    'aoi_disruption_rate': ['aoi_id'],
    'city_hour_rate':      ['city', 'hour_of_day'],
    'courier_city_rate':   ['courier_id', 'city'],
}

def compute_smoothed(agg_df, alpha, global_rate):
    return (agg_df['sum'] + alpha * global_rate) / (agg_df['count'] + alpha)

train_idx = df[is_train].index
test_idx  = df[~is_train].index

for feat in TE_FEATURES:
    df[feat] = np.nan

tscv = TimeSeriesSplit(n_splits=5)
train_pos = np.arange(len(train_idx))

for feat, cols in TE_FEATURES.items():
    oof = pd.Series(index=train_idx, dtype=float)
    for past_pos, current_pos in tscv.split(train_pos):
        fit_rows   = train_idx[past_pos]    
        apply_rows = train_idx[current_pos]   
        agg = df.loc[fit_rows].groupby(cols)[TARGET].agg(['sum','count'])
        rate = compute_smoothed(agg, ALPHA, GLOBAL_RATE)
        keys = df.loc[apply_rows, cols]
        if len(cols) == 1:
            mapped = keys[cols[0]].map(rate)
        else:
            mapped = pd.Series(pd.MultiIndex.from_frame(keys).map(rate), index=apply_rows)
        oof.loc[apply_rows] = mapped.values
    df.loc[train_idx, feat] = oof.values

print("Train expanding-window encoding done.")

for feat, cols in TE_FEATURES.items():
    agg = df.loc[train_idx].groupby(cols)[TARGET].agg(['sum','count'])
    rate = compute_smoothed(agg, ALPHA, GLOBAL_RATE)
    keys = df.loc[test_idx, cols]
    if len(cols) == 1:
        mapped = keys[cols[0]].map(rate)
    else:
        mapped = pd.Series(pd.MultiIndex.from_frame(keys).map(rate), index=test_idx)
    df.loc[test_idx, feat] = mapped.values

print("Test encoding done.")

for feat in TE_FEATURES:
    n_missing = df[feat].isnull().sum()
    df[feat] = df[feat].fillna(GLOBAL_RATE)
    print(f"  {feat:22s} filled {n_missing:>9,} rows with global rate")

print("\nDescribe:")
print(df[list(TE_FEATURES.keys())].describe().T[['mean','min','max']])

Train expanding-window encoding done.
Test encoding done.
  courier_late_rate      filled 1,249,976 rows with global rate
  aoi_disruption_rate    filled   868,109 rows with global rate
  city_hour_rate         filled   824,396 rows with global rate
  courier_city_rate      filled 1,250,000 rows with global rate

Describe:
                         mean       min       max
courier_late_rate    0.018965  0.000048  0.779112
aoi_disruption_rate  0.019833  0.000098  0.448606
city_hour_rate       0.019321  0.000769  0.270073
courier_city_rate    0.018960  0.000048  0.766649


In [ ]:
LEAKAGE_COLS = ['pickup_time', 'pickup_gps_lng', 'pickup_gps_lat',
                'delay_minutes', 'is_disrupted']
print("Leakage columns that must NEVER be features:")
print(f"  {LEAKAGE_COLS}\n")

print(f"courier_orders_so_far min (must be 0): {df['courier_orders_so_far'].min()}")

te_feats = ['courier_late_rate','aoi_disruption_rate','city_hour_rate','courier_city_rate']
print("\nCorrelation of TE features with target (should be modest, NOT ~1.0):")
for f in te_feats:
    c = df[f].corr(df['is_disrupted'])
    print(f"  {f:22s} corr={c:.4f}")

print("\nTrain vs Test mean of each TE feature (should be similar, not identical):")
for f in te_feats:
    tr = df[is_train][f].mean()
    te = df[~is_train][f].mean()
    print(f"  {f:22s} train={tr:.4f}  test={te:.4f}")

ALL_FEATURES = ['hour_of_day','expected_duration','courier_orders_so_far',
                'day_of_week','accept_distance_km'] + te_feats
print(f"\nNaN check across all features:")
print(df[ALL_FEATURES].isnull().sum())

Leakage columns that must NEVER be features:
  ['pickup_time', 'pickup_gps_lng', 'pickup_gps_lat', 'delay_minutes', 'is_disrupted']

courier_orders_so_far min (must be 0): 0

Correlation of TE features with target (should be modest, NOT ~1.0):
  courier_late_rate      corr=0.1737
  aoi_disruption_rate    corr=0.1221
  city_hour_rate         corr=0.0498
  courier_city_rate      corr=0.1731

Train vs Test mean of each TE feature (should be similar, not identical):
  courier_late_rate      train=0.0189  test=0.0190
  aoi_disruption_rate    train=0.0200  test=0.0191
  city_hour_rate         train=0.0195  test=0.0185
  courier_city_rate      train=0.0189  test=0.0190

NaN check across all features:
hour_of_day                    0
expected_duration              0
courier_orders_so_far          0
day_of_week                    0
accept_distance_km       2746705
courier_late_rate              0
aoi_disruption_rate            0
city_hour_rate                 0
courier_city_rate              0


In [ ]:
MODEL_FEATURES = [
    'hour_of_day', 'expected_duration', 'courier_orders_so_far',
    'day_of_week', 'accept_distance_km',
    'courier_late_rate', 'aoi_disruption_rate',
    'city_hour_rate', 'courier_city_rate',
    'aoi_type',
]

KEEP_FOR_SPLIT = ['order_id', 'ds', 'city', 'is_disrupted']

final_cols = KEEP_FOR_SPLIT + MODEL_FEATURES
df_features = df[final_cols].copy()

df_features['city'] = df_features['city'].astype('category')

print(f"Final feature set: {len(MODEL_FEATURES)} features")
print(f"Shape: {df_features.shape}")
print(f"\nColumns: {list(df_features.columns)}")
print(f"\nDtypes:\n{df_features.dtypes}")

out_path = "../../data/processed/pickup_features.parquet"
df_features.to_parquet(out_path, index=False)
print(f"\nSaved → {out_path}")
print(f"File size: {Path(out_path).stat().st_size/1e6:.1f} MB")

Final feature set: 10 features
Shape: (6064908, 14)

Columns: ['order_id', 'ds', 'city', 'is_disrupted', 'hour_of_day', 'expected_duration', 'courier_orders_so_far', 'day_of_week', 'accept_distance_km', 'courier_late_rate', 'aoi_disruption_rate', 'city_hour_rate', 'courier_city_rate', 'aoi_type']

Dtypes:
order_id                    int64
ds                          int64
city                     category
is_disrupted                int64
hour_of_day                 int32
expected_duration         float64
courier_orders_so_far       int64
day_of_week                 int32
accept_distance_km        float64
courier_late_rate         float64
aoi_disruption_rate       float64
city_hour_rate            float64
courier_city_rate         float64
aoi_type                    int64
dtype: object

Saved → ../../data/processed/pickup_features.parquet
File size: 96.4 MB


In [9]:
print("Current df shape:", df.shape)
print("\nColumns available:")
print(list(df.columns))

Current df shape: (6064908, 28)

Columns available:
['order_id', 'city', 'courier_id', 'accept_time', 'time_window_start', 'time_window_end', 'lng', 'lat', 'aoi_id', 'aoi_type', 'pickup_time', 'pickup_gps_lng', 'pickup_gps_lat', 'accept_gps_lng', 'accept_gps_lat', 'ds', 'delay_minutes', 'is_disrupted', 'real_date', 'hour_of_day', 'expected_duration', 'courier_orders_so_far', 'day_of_week', 'accept_distance_km', 'courier_late_rate', 'aoi_disruption_rate', 'city_hour_rate', 'courier_city_rate']


In [ ]:
df['accept_dt'] = df['real_date'] + (df['accept_time'] - df['accept_time'].dt.normalize())

df = df.sort_values(['courier_id', 'aoi_id', 'accept_dt']).reset_index(drop=True)
df['courier_zone_familiarity'] = df.groupby(['courier_id', 'aoi_id']).cumcount()

courier_first = df.groupby('courier_id')['real_date'].transform('min')
df['courier_tenure_days'] = (df['real_date'] - courier_first).dt.days

df = df.sort_values(['courier_id', 'accept_dt']).reset_index(drop=True)
helper = df[['courier_id', 'accept_dt']].copy()
helper['orig_idx'] = df.index
helper = helper.set_index('accept_dt')
def roll_count(g):
    s = g['orig_idx'].rolling('3h', closed='left').count()
    return pd.Series(s.values, index=g['orig_idx'].values)
load_series = helper.groupby('courier_id', group_keys=False).apply(roll_count)

df['courier_load_3h'] = load_series.reindex(df.index).fillna(0).values

df['mins_since_last_accept'] = (
    df.groupby('courier_id')['accept_dt'].diff().dt.total_seconds() / 60
).clip(upper=240).fillna(240)

df['velocity_target'] = df['expected_duration'] / (df['accept_distance_km'] + 0.1)

df = df.sort_values(['courier_id', 'ds', 'accept_dt']).reset_index(drop=True)
df['courier_orders_so_far'] = df.groupby(['courier_id', 'ds']).cumcount()
df['is_first_order_of_day'] = (df['courier_orders_so_far'] == 0).astype(int)

df['is_rush'] = df['hour_of_day'].between(7, 9).astype(int)
df['is_far'] = (df['accept_distance_km'] > 2).astype(int)

print("Group A built. Checks:")
for c in ['courier_zone_familiarity','courier_tenure_days','courier_load_3h',
          'mins_since_last_accept','velocity_target','is_first_order_of_day','is_rush','is_far']:
    print(f"  {c:28s} nulls={df[c].isnull().sum():>9,}  min={df[c].min():.2f}  max={df[c].max():.2f}")

Group A built. Checks:
  courier_zone_familiarity     nulls=        0  min=0.00  max=6845.00
  courier_tenure_days          nulls=        0  min=0.00  max=183.00
  courier_load_3h              nulls=        0  min=0.00  max=77.00
  mins_since_last_accept       nulls=        0  min=0.00  max=240.00
  velocity_target              nulls=2,746,705  min=1.20  max=8507.17
  is_first_order_of_day        nulls=        0  min=0.00  max=1.00
  is_rush                      nulls=        0  min=0.00  max=1.00
  is_far                       nulls=        0  min=0.00  max=1.00


In [ ]:
GLOBAL_RATE = df[df['ds'] <= 950]['is_disrupted'].mean()
ALPHA = 20
print(f"Global rate (train): {GLOBAL_RATE:.5f}, alpha={ALPHA}")

df = df.sort_values(['courier_id', 'accept_dt']).reset_index(drop=True)
total_disrupt_c = df.groupby('courier_id')['is_disrupted'].cumsum()
past_disrupt_c = total_disrupt_c - df['is_disrupted']
past_count_c = df.groupby('courier_id').cumcount()
df['courier_running_rate'] = (past_disrupt_c + ALPHA * GLOBAL_RATE) / (past_count_c + ALPHA)

df = df.sort_values(['aoi_id', 'accept_dt']).reset_index(drop=True)
total_disrupt_z = df.groupby('aoi_id')['is_disrupted'].cumsum()
past_disrupt_z = total_disrupt_z - df['is_disrupted']
past_count_z = df.groupby('aoi_id').cumcount()
df['zone_running_rate'] = (past_disrupt_z + ALPHA * GLOBAL_RATE) / (past_count_z + ALPHA)

print("Running rates built.")
print(f"courier_running_rate: min={df['courier_running_rate'].min():.5f} max={df['courier_running_rate'].max():.5f} nulls={df['courier_running_rate'].isnull().sum()}")
print(f"zone_running_rate:    min={df['zone_running_rate'].min():.5f} max={df['zone_running_rate'].max():.5f} nulls={df['zone_running_rate'].isnull().sum()}")

Global rate (train): 0.01825, alpha=20
Running rates built.
courier_running_rate: min=0.00005 max=0.78514 nulls=0
zone_running_rate:    min=0.00008 max=0.52413 nulls=0


In [ ]:
train_mask = df['ds'] <= 950

daily_cong = (df[train_mask]
              .groupby(['city', 'hour_of_day', 'ds'], observed=True)
              .agg(n_orders=('order_id','size'), n_couriers=('courier_id','nunique'))
              .reset_index())
daily_cong['opc'] = daily_cong['n_orders'] / daily_cong['n_couriers']

cong_profile = (daily_cong.groupby(['city','hour_of_day'], observed=True)['opc']
                .mean().reset_index()
                .rename(columns={'opc':'orders_per_courier'}))

cong_profile['city'] = cong_profile['city'].astype(str)
df['_city_str'] = df['city'].astype(str)

df = df.merge(
    cong_profile.rename(columns={'city':'_city_str'}),
    on=['_city_str','hour_of_day'], how='left'
)
df = df.drop(columns=['_city_str'])

global_opc = daily_cong['opc'].mean()
n_missing = df['orders_per_courier'].isnull().sum()
df['orders_per_courier'] = df['orders_per_courier'].fillna(global_opc)

print(f"orders_per_courier built via merge (categorical-safe)")
print(f"Filled {n_missing:,} unseen rows with global ({global_opc:.3f})")
print(df['orders_per_courier'].describe(percentiles=[.5,.9,.99]).to_string())

orders_per_courier built via merge (categorical-safe)
Filled 0 unseen rows with global (2.532)
count    6.064908e+06
mean     4.614882e+00
std      2.300870e+00
min      1.000000e+00
50%      4.993534e+00
90%      8.020200e+00
99%      8.057338e+00
max      8.057338e+00


In [ ]:
NEW_FEATURES = [
    'courier_zone_familiarity', 'courier_tenure_days', 'courier_load_3h',
    'mins_since_last_accept', 'velocity_target', 'is_first_order_of_day',
    'is_rush', 'is_far',
    'courier_running_rate', 'zone_running_rate', 'orders_per_courier',
]
print("All new features — null check:")
print(df[NEW_FEATURES].isnull().sum())
print(f"\ndf shape: {df.shape}")

All new features — null check:
courier_zone_familiarity          0
courier_tenure_days               0
courier_load_3h                   0
mins_since_last_accept            0
velocity_target             2746705
is_first_order_of_day             0
is_rush                           0
is_far                            0
courier_running_rate              0
zone_running_rate                 0
orders_per_courier                0
dtype: int64

df shape: (6064908, 40)


In [ ]:
print("Correlation of new target-based features with is_disrupted:")
for f in ['courier_running_rate', 'zone_running_rate', 'orders_per_courier']:
    c = df[f].corr(df['is_disrupted'])
    print(f"  {f:25s} corr={c:.4f}")

print("\nFor reference, leakage would show corr > 0.85. Healthy is < 0.4.")

is_train = df['ds'] <= 950
print("\nTrain vs test means (should be similar):")
for f in ['courier_running_rate', 'zone_running_rate', 'orders_per_courier']:
    tr = df[is_train][f].mean()
    te = df[~is_train][f].mean()
    print(f"  {f:25s} train={tr:.4f}  test={te:.4f}")

Correlation of new target-based features with is_disrupted:
  courier_running_rate      corr=0.2869
  zone_running_rate         corr=0.1871
  orders_per_courier        corr=0.0315

For reference, leakage would show corr > 0.85. Healthy is < 0.4.

Train vs test means (should be similar):
  courier_running_rate      train=0.0201  test=0.0196
  zone_running_rate         train=0.0201  test=0.0192
  orders_per_courier        train=4.6230  test=4.5788


In [ ]:
ALL_FEATURES = [
    'hour_of_day', 'expected_duration', 'courier_orders_so_far',
    'day_of_week', 'accept_distance_km',
    'courier_late_rate', 'aoi_disruption_rate', 'city_hour_rate', 'courier_city_rate',
    'aoi_type',
    'courier_zone_familiarity', 'courier_tenure_days', 'courier_load_3h',
    'mins_since_last_accept', 'velocity_target', 'is_first_order_of_day',
    'is_rush', 'is_far',
    'courier_running_rate', 'zone_running_rate', 'orders_per_courier',
]
KEEP = ['order_id', 'ds', 'city', 'is_disrupted']

df_features = df[KEEP + ALL_FEATURES].copy()
df_features['city'] = df_features['city'].astype('category')

out_path = "../../data/processed/pickup_features_v2.parquet"
df_features.to_parquet(out_path, index=False)

print(f"Saved → {out_path}")
print(f"Shape: {df_features.shape}")
print(f"Total features: {len(ALL_FEATURES)}")
print(f"File size: {Path(out_path).stat().st_size/1e6:.1f} MB")

Saved → ../../data/processed/pickup_features_v2.parquet
Shape: (6064908, 25)
Total features: 21
File size: 227.4 MB
